# SignalPilot SDK (`signalpilot-sp`)

**Governed data access for notebooks and scripts.**

The `sp` SDK is a lightweight Python package (zero external dependencies) that connects to the SignalPilot gateway. It provides a clean, Pythonic interface to query databases, explore schemas, and understand data relationships — all through SignalPilot's governance layer.

---

## Install

```bash
pip install signalpilot-sp
```

## 1. Initialization

The SDK supports two modes:

| Mode | Auth | Usage |
|------|------|-------|
| **Local** | None (same as MCP) | Gateway on localhost — no key needed |
| **Cloud** | `sp_` API key (same key used for MCP) | Remote gateway — key required |

The gateway URL resolves in order: **explicit arg > `SP_GATEWAY_URL` env var > `http://localhost:3300`**

In [ ]:
import sp

# Local mode — no auth needed
sp.init()

# Cloud mode — uses the same API key as MCP
# sp.init(api_key="sp_live_abc123...")

# Or set SP_API_KEY env var and just call sp.init()

## 2. List Available Connections

See which database connections are configured in the gateway.

In [ ]:
# List all connections available in the gateway
sp.connections()

## 3. Connect to a Database

`sp.connect()` returns a `Connection` object — all data operations go through it.

You can have multiple connections open simultaneously.

In [ ]:
db = sp.connect("default")
db

In [ ]:
# Work with multiple connections at once
# analytics = sp.connect("analytics")
# warehouse = sp.connect("warehouse")

---

## Connection Methods

Once connected, here's everything you can do:

### 4. `db.query()` — Execute Governed SQL

Run read-only SQL through SignalPilot's governance layer. Queries are validated, audited, and subject to PII redaction, row limits, and budget controls.

Returns a list of row dictionaries.

In [ ]:
# Basic query
rows = db.query("SELECT * FROM customers LIMIT 5")
rows

In [ ]:
# Control the row limit (default: 1000)
rows = db.query("SELECT * FROM orders WHERE status = 'pending'", row_limit=500)

### 5. `db.tables()` — List Tables

Browse all tables available in the connection. Optionally filter by name.

In [ ]:
# List all tables
db.tables()

In [ ]:
# Filter tables by name
db.tables(filter="order")

### 6. `db.describe()` — Column Details

Get detailed column information for a table: types, statistics, annotations, and PII flags.

In [ ]:
db.describe("customers")

### 7. `db.explain()` — Query Plan & Cost Estimate

See the execution plan and estimated cost of a query *without running it*. Useful for validating expensive queries before execution.

In [ ]:
db.explain("SELECT * FROM orders JOIN customers ON orders.customer_id = customers.id")

### 8. `db.sample_values()` — Preview Column Values

See distinct sample values for a column. Great for understanding enums, categories, or foreign key candidates before writing queries.

In [ ]:
# What values does the "status" column have?
db.sample_values("orders", "status")

In [ ]:
# Control sample size
db.sample_values("customers", "country", limit=20)

### 9. `db.join_path()` — Discover How Tables Connect

Find the foreign key path between two tables. No more guessing join conditions — the SDK resolves it from the schema.

In [ ]:
# How do I join orders to customers?
db.join_path("orders", "customers")

In [ ]:
# Multi-hop joins — find paths through intermediate tables
db.join_path("order_items", "customers", max_hops=4)

### 10. `db.schema_overview()` — High-Level Summary

Get a bird's-eye view of the database: table counts, total rows, and overall structure.

In [ ]:
db.schema_overview()

---

## API Reference

| Method | Description | Returns |
|--------|-------------|---------|
| `sp.init()` | Initialize SDK (local, no auth) | `None` |
| `sp.init(api_key="sp_...")` | Initialize SDK (cloud, with API key) | `None` |
| `sp.connections()` | List available connection names | `list[str]` |
| `sp.connect(name)` | Get a Connection object | `Connection` |
| `db.query(sql, row_limit=1000)` | Execute governed SQL | `list[dict]` |
| `db.tables(filter=None)` | List tables | `list[dict]` |
| `db.describe(table)` | Column details, types, stats | `list[dict]` |
| `db.explain(sql)` | Query plan and cost estimate | `dict` |
| `db.sample_values(table, column, limit=50)` | Distinct sample values | `list` |
| `db.join_path(from_table, to_table, max_hops=4)` | FK join path discovery | `list[dict]` |
| `db.schema_overview()` | High-level schema summary | `dict` |

## Environment Variables

| Variable | Purpose | Default |
|----------|---------|---------|
| `SP_GATEWAY_URL` | Gateway URL | `http://localhost:3300` |
| `SP_API_KEY` | API key for cloud (same as MCP) | — |

## What Happens Under the Hood

Every call through the SDK goes through the SignalPilot gateway, which enforces:

- **SQL validation** — blocks DDL/DML, statement stacking, and restricted tables/columns
- **Row limits** — auto-injected LIMIT clauses, configurable per query
- **PII redaction** — sensitive columns masked, hashed, or hidden based on connection rules
- **Audit trail** — every query logged with user, timestamp, and connection
- **Budget controls** — per-session cost tracking and limits
- **Query cost estimation** — estimated cost via EXPLAIN before execution

The SDK itself is stateless and stdlib-only (zero external dependencies). It uses the same `sp_` API keys as the MCP tools — one credential for everything.